<a href="https://colab.research.google.com/github/Teivak/FaceRecognitionProject/blob/main/3_HW_ArcFace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ArcFace Loss (Additive Angular Margin Loss)

## Теория ArcFace

В случае с обучением на задачу классификации первая подходящая лосс-функция, которая нам приходит в голову — Cross-Entropy. И на ней действительно можно обучать сеть для распознавания лиц. Но за много лет люди придумали более хитрые трюки, которые делают обучение сети для распознавания лиц более эффективным. Одним из лучших подходов считается ArcFace (Additive Angular Margin).


**Как устроен ArcFace**:

Стандартные SoftMax + кросс-энтропия (CE) выглядят так:

$$L_{CE} = \frac{-1}{N}\sum_1^N \frac{e^{W_{y_i}^{T}x_i + b_{y_i}}}{\sum^n_{j=1}e^{W_j^Tx_i+b_j}},$$

здесь:
- $x_i \in \mathbb{R^d}$ — вектор $i$-го элемента обучающей выборки перед последним полносвязным слоем сети. $y_i$ — класс этого элемента;
- $W_j \in \mathbb{R^d}$ — j-ый столбец матрицы весов последнего слоя сети (т.е. слоя, который производит итоговую классификацю входящего объекта);
- $b_j \in \mathbb{R^d}$ — j-ый элемент вектора байеса последнего слоя сети;
- $N$ — batch size;
- $n$ — количество классов.


Хотя этот лосс работает хорошо, он явным образом не заставляет эмбеддинги $x_i$ элементов, принадлежащих одному классу, быть близкими друг к другу по расстоянию. И не заставляет эмбеддинги элементов, принадлежащих разным классам, быть далеко друг от друга. Все, что хочет этот лосс — чтобы на основе эмбеддингов $x_i$ можно было хорошо классифицировать элементы, никакие ограничений на расстояния между эмбеддингами $x_i$ он не вводит.

Из-за этого у нейросетей для распознавания лиц, которые обучены на обычном CE loss, бывают проблемы с распознаванием лиц, которые сильно отличаются от лиц того же человека разными доп. атрибутами (шляпа/прическа/очки и т.п.). Просто эмбеддинг для таких лиц получается довольно далек по расстоянию от других эмбеддингов лиц этого же человека.

Давайте теперь немного поправим формулу:
- уберем байес последнего слоя, т.е. сделаем $b_j=0$;
- нормализуем веса последнего слоя: ||$W_j$|| = 1;
- нормализуем эмбеддинги: ||$x_i$|| = 1. Перед подачей их на вход последнему слою (т.е. перед умножением на матрицу $W_j$) умножим их на гиперпараметр s. По сути, мы приводим норму всех эмбеддингов к s. Смысл этого гиперпараметра в том, что, возможно, сети проще будет классифицировать эмбеддинги, у которых не единичная норма.

Нормализация приводит к тому, что эмбеддинги распределяются по сфере единичного радиуса (и сфере радиуса s после умножения на гиперпараметр s). И итоговые предсказания сети после последнего слоя зависят только от угла между эмбеддингами $x_i$ и выученных весов $W_j$. От нормы эмбеддинга $x_i$ они больше не зависят, т.к. у всех эмбеддингов они теперь одинаковые.

Получается, в степени экспоненты у нас останется выражение $s W_{y_i}^{T}x_i$, которое можно переписать в виде  $s W_{y_i}^{T}x_i = s ||W_{y_i}||\cdot ||x_i|| \cdot cos\Theta_{y_i}$. Тут $\Theta_{y_i}$ — это угол между векторами $W_{y_i}$ и $x_i$. Но так как мы сделали нормы $W_{y_i}$ и $x_i$ единичными, то все это выражение просто будет равно $s cos\Theta_{y_i}$.

В итоге мы получим следующую формулу лосса:

$$L = \frac{-1}{N}\sum_1^N \frac{e^{s\ cos\Theta_{y_i}}}{e^{s\ cos\Theta_{y_i}} + \sum^n_{j=1,\ j\ne y_i} e^{s\ cos\Theta_j}}$$


И последний шаг. Добавим еще один гиперпараметр $m$. Он называется additive angular margin penalty и заставляет эмбеддинги одного класса быть ближе друг к другу, а эмбеддинги разных классов — более далекими друг от друга.

В итоге получим вот что:

$$L_{ArcFace} = \frac{-1}{N}\sum_1^N \frac{e^{s\ cos(\Theta_{y_i} + m)}}{e^{s\ cos(\Theta_{y_i} + m)} + \sum^n_{j=1,\ j\ne y_i} e^{s\ cos\Theta_j}}$$

Это и есть ArcFace Loss с двумя  гиперпараметрами, s и m.

Получается, что ArcFace Loss завтавляет сеть выучивать эмбеддинги, распределенные по сфере радиуса s, причем чтобы эмбеддинги одного класса были ближе друг к другу, а эмбеддинги разных классов — более далеки друг от друга.

![ArcFace](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTKRR-YA_XR3yhIYBbkc8Zlbua0Q2WdM3gx_g&s)

**Важное пояснение:**

Строго говоря, ArcFace - не лосс, отдельный архитектурный модуль модификация SoftMax. Он реализует идею внесения геометрического отступа непосредственно в пространство признаков. Для обучения в качестве лосса используется обычная кросс-энтропия. Более конкретно по шагам:

1. Вы извлекаете эмбеддинги из бэкбона сети (предобученной модели, у которой обрезан FC-слой, если он был)
2. Эти эмбеддинги поступают в ArcFace-слой, который содержит векторы-центры для каждого класса (веса классификатора) и логику нормализации и добавления углового отступа
3. Для целевого класса ArcFace-слой преобразует косинус угла $\theta$ в $cos(\theta + m)$
4. Для остальных классов оставляет обычный косинус $cos(\theta)$
5. Эти модифицированные логиты подаются на вход стандартной функции Cross-Entropy
6. Градиенты от Cross-Entropy текут назад через ArcFace-слой к бэкбону, обучая модель извлекать эмбеддинги

Результат: модифицированные логиты с "жестким" разделением для целевого класса, а значит и более качественные эмбеддинги.

Схема:
```
[Изображение] → [Бэкбон] → [ЭМБЕДДИНГ] → [ArcFace] → [Логиты] → [CE Loss]
                    │                        │           │          
                   CNN                   Нормализация   Оценки
                                          + Angular    для всех
                                            Margin     классов
```

Для получения качественных эмбеддингов после обучения ArcFace-слой больше не нужен, и его обычно обрезают. Он нужен был только обучения модели, и поэтому часто ArcFace называю именно лоссом. Но стоит всегда держать в голове, что это некоторое упрощение, которое нужно лишь для того, чтобы проще формулировать мысли.

**Доп. литература по ArcFace Loss:**

Оригинальная статья: https://arxiv.org/pdf/1801.07698.pdf

## Другие лоссы

Кроме ArcFace, есть еще много разных вариантов лоссов для задачи Face Recognition. Некоторые из них можно найти, например, [тут](https://openaccess.thecvf.com/content_CVPRW_2020/papers/w48/Hsu_A_Comprehensive_Study_on_Loss_Functions_for_Cross-Factor_Face_Recognition_CVPRW_2020_paper.pdf). Вы можете попробовать реализовать другие лосс-функции в этом проекте в качестве дополнительного задания.

Кроме этого, можно миксовать лосс-функции. Например, обучать нейросеть на сумме ArcFace и TripletLoss. Иногда так выходит лучше, чем если обучать на каком-то одном лоссе.

# Датасет

В качестве датасета нужно использовать картинки из CelebA, выровненные при помощи своей модели из задания 1. Очень желательно их еще кропнуть таким образом, чтобы нейросети поступали на вход преимущественно только лица без какого либо фона, частей тела и прочего.

Если планируете делать дополнительное задание на Identificaton rate metric, то **обязательно разбейте заранее датасет на train/val или train/val/test.** Это нужно сделать не только на уровне кода, а на уровне папок, чтобы точно знать, на каких картинках модель обучалась, а на каких нет. Лучше заранее почитайте [ноутбук с заданием](https://colab.research.google.com/drive/15zuNdOupRFnG7oE-rFj9FsjoNTK6DYn5).

# План заданий

Итак, вот, что от вас требуется в этом задании:

* Выбрать модель (или несколько моделей) для обучения. Можно брать предобученные на ImageNet, но нельзя использовать модели, предобученные на задачу распознавания лиц.
* Обучить эту модель (модели) на CE loss. Добиться accuracy > 0.7.
* Реализовать ArcFace loss.
* Обучить модель (модели) на ArcFace loss. Добиться accuracy > 0.7.
* Написать небольшой отчет по обучению, сравнить CE loss и ArcFace loss.

**P.S. Не забывайте сохранять модели после обучения**

In [ ]:
%%writefile PROJECT/FaceAlignment/FaceRecognitionDataset.py

import numpy as np
import os
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T
from torchvision.tv_tensors import KeyPoints
import cv2 # Импортируем cv2 для функции face_align
from PROJECT.FaceAlignment.face_align import face_align # Добавляем явный импорт face_align
from sklearn.preprocessing import LabelEncoder # Импортируем LabelEncoder

# Мы сохраняем этот блок кода отдельно, так что желательно,
# чтобы все нужные библиотеки, а также path были доступны
path = '/home/timof/.cache/kagglehub/datasets/kevinpatel04/celeba-original-wild-images/versions/1' # This 'path' is passed as images_path to the dataset
aligned_images_dir = 'PROJECT/FaceAlignment/aligned_images'

def create_heatmap(size, landmark, sigma=2):
    """
    Создаёт один heatmap с гауссовым ядром вокруг точки.

    :param size: (height, width) — размер heatmap'а
    :param landmark:(x, y) — координаты точки
    :param sigma
    :return: heatmap массив
    """
    x, y = landmark
    h, w = size

    # Обрезаем координаты, чтобы не выйти за пределы изображения
    x = min(max(0, int(x)), w - 1)
    y = min(max(0, int(y)), h - 1)

    xx, yy = np.meshgrid(np.arange(w), np.arange(h))
    heatmap = np.exp(-((yy - y)**2 + (xx - x)**2) / (2 * sigma**2))
    return heatmap


def landmarks_to_heatmaps(image_shape, landmarks, sigma=2):
    """
    Преобразует список из N точек в набор из N heatmap'ов.

    :param image_shape: исходный размер изображения (H, W)
    :param landmarks: список из N пар координат [(x1, y1), (x2, y2), ..., (xN, yN),]
    :param sigma:
    :return: массив heatmap'ов вида [N, H, W]
    """
    heatmaps = []

    for (x, y) in landmarks:
        hm = create_heatmap(image_shape, landmark=(x,y), sigma=sigma)
        heatmaps.append(hm)

    return np.array(heatmaps)


# Функция для ресайза изображений с сохранением соотношений сторон
# Пустое пространство заполняется чёрным с помощью Pad
# При этом, она ещё и адаптирует положение лэндмарков, используя torchvision.tv_tensors.KeyPoints
class ResizeAndPad(T.Transform):
    def __init__(self, output_size=(128, 128)):
        super().__init__()
        assert isinstance(output_size, (int, tuple))
        if isinstance(output_size, int):
            self.output_size = (output_size, output_size)
        else:
            assert len(output_size) == 2
            self.output_size = output_size

    def forward(self, data): # Now expects tv_tensors.Image and KeyPoints
        image = data['image'] # Expects tv_tensors.Image
        # Keypoints are already tv_tensors.KeyPoints and will be handled by T.Resize/T.Pad if in data dict

        w, h = image.shape[-2], image.shape[-1] # Assuming CHW format for tv_tensors.Image
        target_w, target_h = self.output_size
        # Вычисляем коэффициент масштабирования, чтобы вписать изображение в target_size, сохраняя соотношение сторон
        scale = min(target_w / w, target_h / h)
        new_w, new_h = int(w * scale), int(h * scale)

        # Изменяем размер данных (изображение и ключевые точки). T.Resize автоматически обрабатывает tv_tensors.
        # Используем список для size=(new_h, new_w), так как T.Resize ожидает последовательность.
        data = T.Resize(size=(new_w, new_h), interpolation=T.InterpolationMode.BICUBIC)(data)

        # Пересчитываем размеры для заполнения после изменения размера, так как фактические new_w, new_h могут незначительно отличаться из-за преобразования в int()
        current_h, current_w = data['image'].shape[-2], data['image'].shape[-1] # Shape is (C, H, W)

        # Вычисляем размеры отступов для current_w, current_h, чтобы достичь target_w, target_h
        pad_w = target_w - current_w
        pad_h = target_h - current_h
        # T.Pad expects (padding_left, padding_top, padding_right, padding_bottom)
        padding = (pad_w // 2, pad_h // 2, pad_w - pad_w // 2, pad_h - pad_h // 2)
        # Применяем заполнение к данным. T.Pad автоматически обрабатывает tv_tensors.
        data = T.Pad(padding=padding, fill=0)(data)

        return data # Возвращает трансформированное изображение и адаптированные под него лэндмарки


# Функция для выравнивания изображений
# Она использует функцию face_align, которая будет написана ближе к концу ноутбука
class FaceAlign(T.Transform):
    def __init__(self, target_size=(128, 128), target_left_eye=(0.28, 0.35), target_right_eye=(0.72, 0.35), target_mouth_y=0.75):
        super().__init__()
        assert isinstance(target_size, (int, tuple))
        if isinstance(target_size, int):
            self.target_size = (target_size, target_size)
        else:
            assert len(target_size) == 2
            self.target_size = target_size
        self.target_left_eye = target_left_eye
        self.target_right_eye = target_right_eye
        self.target_mouth_y = target_mouth_y

    def forward(self, data): # Now expects tv_tensors.Image and KeyPoints
        image_tensor = data['image'] # Expects tv_tensors.Image (Tensor, CHW)
        keypoints_obj = data['keypoints'] # This should be tv_tensors.KeyPoints
        # Convert tv_tensors.Image (CHW, float32) to NumPy (HWC, float32 or uint8) for face_align function
        # Assuming face_align works with float32 [0,1] input. If it expects uint8, an explicit conversion is needed.
        image_np = image_tensor.permute(1, 2, 0).cpu().numpy()

        # Extract landmarks from KeyPoints object (convert to NumPy)
        landmarks_np = keypoints_obj.data.cpu().numpy() # Shape (N, 2)

        # Вызываем функцию face_align
        aligned_face_np, M = face_align(image_np,
                                        landmarks_np,
                                        target_size=self.target_size,
                                        target_left_eye=self.target_left_eye,
                                        target_right_eye=self.target_right_eye,
                                        target_mouth_y=self.target_mouth_y)

        # Convert aligned NumPy array back to tv_tensors.Image
        # T.ToImage() expects HWC numpy array. It will handle float32 [0,1] or uint8 [0,255].
        aligned_image_tensor = T.ToImage()(aligned_face_np)
        float_image_tensor = T.ConvertImageDtype(torch.float)(aligned_image_tensor)
        # Dummy keypoints, canvas size matches target_size as it's an aligned image
        dummy_keypoints = KeyPoints(torch.empty(0, 2), canvas_size=self.target_size)

        # Возвращаем трансформированное изображение (тензор) и ключевые точки
        return {'image': float_image_tensor.to(device), 'keypoints': dummy_keypoints}



class FaceRecognitionDataset(Dataset):
    def __init__(self, df, images_path, aligned_images_dir, target_image_size=(128, 128), augment_transform=False, mode='landmark_prediction'):
        self.df = df
        self.images_path = images_path
        self.aligned_images_dir = aligned_images_dir
        self.target_image_size = target_image_size
        self.mode = mode
        self.num_classes = None

        if mode == 'face_recognition':
            # Only fit a new LabelEncoder if one is not provided (for training dataset)
            all_person_ids = self.df['person_id'].unique()
            all_person_ids_sorted = np.sort(all_person_ids) # Ensure consistent encoding
            self.label_encoder = LabelEncoder()
            self.label_encoder.fit(all_person_ids_sorted)
            # Store num_classes derived from the (provided or newly fitted) encoder
            self.num_classes = len(self.label_encoder.classes_)

        self.base_transform = T.Compose([
                T.PILToTensor(),
                T.ConvertImageDtype(torch.float)
                ])

        # Добавляем аугментации, если augment_transform предоставлен, или используем набор по умолчанию
        if augment_transform:
            # Аугментации по умолчанию (для обучения распознавателя позже)
            augment_transforms = [
                T.RandomRotation(5),
                T.RandomApply([T.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0))], p=0.2), # Гауссовский шум
                T.RandomApply([T.GaussianNoise(sigma=0.08)], p=0.2), # Гауссовский шум
                T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                T.RandomGrayscale(0.2),
                T.RandomHorizontalFlip(p=0.5)
            ]
        else:
            augment_transforms = []

        # Объединяем для конечного пайплайна
        self.transform = T.Compose(augment_transforms + [
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])


    def bbox_crop(self, row, data):
        # Получаем исходные размеры ббокса
        x1_orig, y1_orig = row['x_1'], row['y_1']
        cropped_width, cropped_height = row['width'], row['height']

        if cropped_width == 0 or cropped_height == 0:
            print(f"Предупреждение: Пропускаем {row['image_id']} из-за недопустимых ориентиров (нулевая ширина/высота bbox).")
            return None

        # Extract image and keypoints from the dictionary
        image = data['image']
        keypoints = data.get('keypoints')

        # Apply crop to the image
        cropped_image = T.functional.crop(image, y1_orig, x1_orig, cropped_height, cropped_width)

        # Create a new dictionary for the output
        result_data = {'image': cropped_image}

        # If keypoints were present, crop them and add to the result
        if keypoints is not None:
            cropped_keypoints = T.functional.crop(keypoints, y1_orig, x1_orig, cropped_height, cropped_width)
            result_data['keypoints'] = cropped_keypoints
        else:
            # If no keypoints were provided, return dummy KeyPoints matching the cropped image size
            result_data['keypoints'] = KeyPoints(torch.empty(0, 2), canvas_size=(cropped_height, cropped_width))

        return result_data

    def __len__(self):
        return len(self.df)

    def _get_image(self, image_id):
        # Получение изображения из исходной папки с каггла
        part = (int(image_id[:-4]) - 1) // 10000 + 1
        image_full_path = os.path.join(self.images_path, f'Part {part}', f'Part {part}', image_id)
        try:
            image = Image.open(image_full_path).convert('RGB')
            return self.base_transform(image).to(device)
        except FileNotFoundError:
            print(f'Путь не найден. Возможно файла "{image_id}" не существует. Пропускаем изображение.')
            return None

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row['image_id']
        person_id = row['person_id']

        aligned_path = os.path.join(self.aligned_images_dir, f'{image_id}.pt')

        if not(self.mode == 'face_recognition' and os.path.exists(aligned_path)):
            # Получаем изображение
            image = self._get_image(image_id)
            if image is None:
                return None

        if self.mode == 'face_recognition':
            # Проверяем, существует ли уже выровненное изображение

            # Ensure aligned_images_dir exists
            os.makedirs(self.aligned_images_dir, exist_ok=True)


            if os.path.exists(aligned_path):
                # Если тензор изображения существует, загружаем его
                aligned_image = torch.load(aligned_path, weights_only=False).to(device)
                # Создаем пустой объект KeyPoints, так как для уже выровненного изображения они не нужны
                dummy_keypoints = KeyPoints(torch.empty(0, 2), canvas_size=self.target_image_size)
                transformed_data = self.transform({'image': aligned_image, 'keypoints': dummy_keypoints})
            else:
                # Image is already a tensor (C, H, W) from _get_image
                orig_h, orig_w = image.shape[-2], image.shape[-1]
                cropped_width, cropped_height = row['width'], row['height']

                # Extract landmarks from the dataframe row
                landmarks_coords = [
                    (row['lefteye_x'], row['lefteye_y']),
                    (row['righteye_x'], row['righteye_y']),
                    (row['nose_x'], row['nose_y']),
                    (row['leftmouth_x'], row['leftmouth_y']),
                    (row['rightmouth_x'], row['rightmouth_y'])
                ]

                # Create KeyPoints object with original image dimensions
                keypoints_obj = KeyPoints(torch.tensor(landmarks_coords, dtype=torch.float),
                                          canvas_size=(cropped_height, cropped_width))

                # Prepare data dictionary for FaceAlign transform
                data = {'image': image}
                cropped_data = self.bbox_crop(row, data)
                # Corrected ResizeAndPad instantiation and application
                resized_data = ResizeAndPad(self.target_image_size)(cropped_data)
                resized_data['keypoints'] = keypoints_obj
                aligned_data = FaceAlign(target_size=self.target_image_size)(resized_data)

                # Save the newly aligned image as a PyTorch tensor
                torch.save(aligned_data['image'], aligned_path)

                # Now apply the remaining transforms (normalization and augmentations)
                transformed_data = self.transform(aligned_data)

            encoded_person_id = self.label_encoder.transform([person_id])[0]
            return transformed_data['image'], encoded_person_id

        elif self.mode == 'landmark_prediction':
            # Image is already a tensor (C, H, W) from _get_image
            orig_h, orig_w = image.shape[-2], image.shape[-1]

            # Extract landmarks from the dataframe row
            landmarks_coords = [
                (row['lefteye_x'], row['lefteye_y']),
                (row['righteye_x'], row['righteye_y']),
                (row['nose_x'], row['nose_y']),
                (row['leftmouth_x'], row['leftmouth_y']),
                (row['rightmouth_x'], row['rightmouth_y'])
            ]

            # Create KeyPoints object with original image dimensions
            # Канвас оригинальных лэндмарков будет размером с исходное изображение
            keypoints_obj = KeyPoints(torch.tensor(landmarks_coords, dtype=torch.float),
                                      canvas_size=(orig_h, orig_w))

            # Prepare data dictionary for bbox_crop transform
            data_for_crop = {'image': image, 'keypoints': keypoints_obj}
            cropped_data = self.bbox_crop(row, data_for_crop)
            # Corrected ResizeAndPad instantiation and application
            resized_data = ResizeAndPad(self.target_image_size)(cropped_data)
            # Apply the transform pipeline to the dictionary containing the image and keypoints
            transformed_data = self.transform(resized_data)

            transformed_keypoints_obj = transformed_data['keypoints']

            # This part remains as original for landmark_prediction mode
            final_landmarks_for_heatmap_and_plotting = transformed_keypoints_obj.data.cpu().numpy().tolist()

            # Определяем целевой размер хитмапы
            heatmap_target_h, heatmap_target_w = 64, 64
            input_image_h, input_image_w = self.target_image_size # (128, 128)

            # Масштабируем ориентиры из размера входного изображения (128x128) к целевому размеру хитмапы (64x64)
            scale_factor_h_for_heatmap = heatmap_target_h / input_image_h
            scale_factor_w_for_heatmap = heatmap_target_w / input_image_w

            scaled_landmarks_for_heatmap = []
            for lx, ly in final_landmarks_for_heatmap_and_plotting:
                scaled_landmarks_for_heatmap.append((int(round(lx * scale_factor_w_for_heatmap)),
                                                     int(round(ly * scale_factor_h_for_heatmap))))

            # Переводим лэндмарки в хитмапы
            heatmaps_np = landmarks_to_heatmaps((heatmap_target_h, heatmap_target_w), scaled_landmarks_for_heatmap)
            heatmaps_tensor = torch.from_numpy(heatmaps_np).float()

            # Также переводим лэндмарки в тензор
            adjusted_landmarks_tensor = torch.tensor(final_landmarks_for_heatmap_and_plotting).float()

            return transformed_data['image'], heatmaps_tensor, adjusted_landmarks_tensor
        else:
            raise ValueError(f"Неизвестный mode: {self.mode}. Выберите 'face_recognition' или 'landmark_prediction'.")

# Function custom_collate_fn for filtering None-samples (remains same)
def custom_collate_fn(batch):
    # Filter None-samples (e.g., failed image loading)
    batch = [item for item in batch if item is not None]
    if not batch:
        # If batch is empty after filtering, return None to be skipped by DataLoader
        return None

    return torch.utils.data.dataloader.default_collate(batch)

In [ ]:
import pandas as pd

train_dataset_df = pd.read_csv('PROJECT/FaceAlignment/train_dataset.csv')
val_dataset_df = pd.read_csv('PROJECT/FaceAlignment/val_dataset.csv')
test_dataset_df = pd.read_csv('PROJECT/FaceAlignment/test_dataset.csv')

In [ ]:
import kagglehub
import os
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("kevinpatel04/celeba-original-wild-images")

landmarks = pd.read_csv('PROJECT/FaceAlignment/pred_landmarks.csv')
bboxes = pd.read_csv(f'{path}/list_bbox_celeba.csv')

# Helper function to merge a dataset dataframe with landmarks and bboxes
def merge_datasets(dataset_df, landmarks_df, bboxes_df):
    merged_df = pd.merge(dataset_df, landmarks_df, on='image_id', how='left')
    merged_df = pd.merge(merged_df, bboxes_df, on='image_id', how='left')
    return merged_df

# Merge train, val, and test dataframes with landmarks and bboxes
full_train_dataset_df = merge_datasets(train_dataset_df, landmarks, bboxes)
full_val_dataset_df = merge_datasets(val_dataset_df, landmarks, bboxes)
full_test_dataset_df = merge_datasets(test_dataset_df, landmarks, bboxes)

In [ ]:
import torch
from torch.utils.data import DataLoader, default_collate
import numpy as np

fixed_image_size = (128, 128) # Определяем фиксированный размер для всех изображений

# For train_dataset, we fit the LabelEncoder
train_dataset = FaceRecognitionDataset(
    full_train_dataset_df,
    path,
    aligned_images_dir=aligned_images_dir,
    augment_transform = True,
    mode='face_recognition',
    target_image_size=fixed_image_size,
)

# For val_dataset, use the filtered dataframe and the label_encoder fitted on the train_dataset
val_dataset = FaceRecognitionDataset(
    full_val_dataset_df, # Use filtered dataframe
    path,
    aligned_images_dir=aligned_images_dir,
    mode='face_recognition',
    target_image_size=fixed_image_size,
)

# For test_dataset, use the filtered dataframe and the label_encoder fitted on the train_dataset
test_dataset = FaceRecognitionDataset(
    full_test_dataset_df, # Use filtered dataframe
    path,
    aligned_images_dir=aligned_images_dir,
    mode='face_recognition',
    target_image_size=fixed_image_size,
)

batch_size = 256
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn, num_workers=0)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn, num_workers=0)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn, num_workers=0)

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights


class FaceRecognitionBackbone(nn.Module):
    def __init__(self, backbone_model=None, embedding_dim=512):
        super(FaceRecognitionBackbone, self).__init__()
        # Load a pre-trained ResNet50 model
        if backbone_model is None:
            self.backbone = resnet50(weights=ResNet50_Weights.DEFAULT)
        else:
            # If a custom backbone is provided, use it
            self.backbone = backbone_model

        if hasattr(self.backbone, 'fc'):
            self.backbone.fc = nn.Identity()
        elif hasattr(self.backbone, 'classifier'): # For models like EfficientNet
            self.backbone.classifier = nn.Identity()

        # Add other conditions here for different model architectures if needed

        # Dynamically infer in_features for embedding_head
        # Create a dummy input to pass through the backbone to get the output shape
        # Defaulting to 224x224, but this might need adjustment for specific models (e.g., EfficientNet-B7 expects 600x600)
        dummy_input = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            # Pass the dummy input through the modified backbone (without the FC layer/classifier)
            # Flatten the output if it's not already 2D (e.g., coming from Conv layers)
            dummy_output = self.backbone(dummy_input)
            # If the output is (1, C, H, W), flatten to (1, C*H*W) before getting size(1)
            if dummy_output.dim() > 2:
                in_features = dummy_output.view(dummy_output.size(0), -1).size(1)
            else:
                in_features = dummy_output.size(1)

        # Add a new fully connected layer for embedding
        self.embedding_head = nn.Linear(in_features, embedding_dim)

        # Add a batch normalization layer after the embedding head
        self.bn = nn.BatchNorm1d(embedding_dim)

    def forward(self, x):
        # Pass input through the ResNet backbone
        x = self.backbone(x)

        # If the output is still a feature map (e.g., from EfficientNet's features), flatten it
        if x.dim() > 2:
            x = x.view(x.size(0), -1)

        # Pass through the embedding head
        x = self.embedding_head(x)

        # Pass through batch normalization
        x = self.bn(x)

        return x

In [ ]:
import math
import torch.nn.functional as F

class ArcFace(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.50):
        super(ArcFace, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m

        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input, label=None):
        # Normalize the input embeddings and weights
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))

        # If label is None, return unmarginalized, scaled logits
        if label is None:
            return cosine * self.s

        sine = torch.sqrt(1.0 - torch.pow(cosine, 2))
        phi = cosine * self.cos_m - sine * self.sin_m

        # Apply margin to target classes
        if self.m > 0:
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        # Clamp labels to ensure they are within the valid range [0, num_classes-1]
        # This prevents F.one_hot from producing an empty tensor if labels are out of bounds.
        labels_clamped = torch.clamp(label, 0, self.out_features - 1)

        one_hot = F.one_hot(labels_clamped, num_classes=self.out_features).float()
        # Combine original logits with marginalized logits for target classes
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)

        # Scale the logits
        output *= self.s

        return output

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class AdaCos(nn.Module):
    def __init__(self, in_features, out_features):
        super(AdaCos, self).__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        # Initialize s dynamically during training, this will be handled in the forward pass.
        self.s = math.sqrt(2) * math.log(out_features - 1)

    def forward(self, input, label=None):
        # Normalize input embeddings and weights
        normalized_input = F.normalize(input)
        normalized_weight = F.normalize(self.weight)

        # Calculate cosine similarity
        cosine = F.linear(normalized_input, normalized_weight)
        if label is None:
            return cosine * self.s

        # Create one-hot labels for sparse target (label) and expand to match cosine shape
        one_hot = F.one_hot(label, num_classes=self.out_features).float()

        # Combine original logits with marginalized logits for target classes
        # For target classes, use cosine_with_margin, for others, use original cosine
        output = (one_hot * cosine) + ((1.0 - one_hot) * cosine)

        # Adaptive scaling factor 's' (AdaCos specific)
        # This part requires specific implementation details of AdaCos, often an adaptive 's' based on cosine values
        # For simplicity, we use a simplified adaptive scaling based on the median of current batch's target cosines.

        with torch.no_grad():
            B_avg = torch.where(one_hot == 1, cosine, torch.zeros_like(cosine))
            B_avg = B_avg.sum(dim=1) / (one_hot.sum(dim=1) + 1e-8) # Add small epsilon to prevent division by zero
            theta_median = torch.acos(torch.clamp(B_avg, -1.0 + 1e-7, 1.0 - 1e-7)).median()
            # Fix: Convert float to tensor for torch.log
            self.s = torch.log(torch.tensor(self.weight.size(0) - 1.0, device=input.device))

        # Scale the logits
        output *= self.s

        return output

In [ ]:
import torch.nn as nn

class LinearClassifier(nn.Module):
    def __init__(self, in_features, out_features):
        super(LinearClassifier, self).__init__()
        self.fc = nn.Linear(in_features, out_features)

    def forward(self, input):
        # For standard linear classification, we just pass the embeddings through a linear layer.
        # Normalization is not typically required here as it's often handled before this layer
        # or by the loss function itself (e.g., CrossEntropyLoss expects raw logits).
        logits = self.fc(input)
        return logits

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights # Import EfficientNet

# Ensure num_classes is available from previous steps (it was defined as 1944)
# Ensure FaceRecognitionBackbone and LinearClassifier classes are available from previous steps

class FR_LinearEfficientNetModel(nn.Module):
    def __init__(self, embedding_dim=1280, num_classes=None):
        super(FR_LinearEfficientNetModel, self).__init__() # Updated super call
        if num_classes is None:
            raise ValueError("num_classes must be provided for FaceRecognitionEfficientNetModel")

        # Initialize EfficientNet-B0 as the backbone
        efficientnet_backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

        # Pass the EfficientNet backbone to our generic FaceRecognitionBackbone wrapper
        self.backbone = FaceRecognitionBackbone(backbone_model=efficientnet_backbone, embedding_dim=embedding_dim)
        self.classifier = LinearClassifier(in_features=embedding_dim, out_features=num_classes)

    def forward(self, x):
        embeddings = self.backbone(x)
        logits = self.classifier(embeddings)
        return logits

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights # Import EfficientNet

# Ensure num_classes is available from previous steps (it was defined as 1944)
# Ensure FaceRecognitionBackbone and LinearClassifier classes are available from previous steps

class FR_ArcFaceEfficientNetModel(nn.Module):
    def __init__(self, embedding_dim=1280, num_classes=None, s=64.0, m=0.50):
        super(FR_ArcFaceEfficientNetModel, self).__init__() # Updated super call
        if num_classes is None:
            raise ValueError("num_classes must be provided for FaceRecognitionEfficientNetModel")

        # Initialize EfficientNet-B0 as the backbone
        efficientnet_backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

        # Pass the EfficientNet backbone to our generic FaceRecognitionBackbone wrapper
        self.backbone = FaceRecognitionBackbone(backbone_model=efficientnet_backbone, embedding_dim=embedding_dim)
        self.arcface = ArcFace(in_features=embedding_dim, out_features=num_classes, s=s, m=m)

    def forward(self, x, label=None):
        embeddings = self.backbone(x)
        logits = self.arcface(embeddings, label)
        return logits

In [ ]:
import torch.nn as nn

# Ensure num_classes is available from previous steps (it was defined as 1944)
# Ensure FaceRecognitionBackbone and LinearClassifier classes are available from previous steps

class FaceRecognitionModel(nn.Module):
    def __init__(self, embedding_dim=512, num_classes=None):
        super(FaceRecognitionModel, self).__init__()
        if num_classes is None:
            raise ValueError("num_classes must be provided for FaceRecognitionModel")
        self.backbone = FaceRecognitionBackbone(embedding_dim=embedding_dim)
        self.classifier = LinearClassifier(in_features=embedding_dim, out_features=num_classes)

    def forward(self, x):
        embeddings = self.backbone(x)
        logits = self.classifier(embeddings)
        return logits

In [ ]:
import torch.nn as nn

# Ensure num_classes is available from previous steps (it was defined as 1944)
# Ensure FaceRecognitionBackbone and ArcFace classes are available from previous steps

class FaceRecognitionArcFaceModel(nn.Module):
    def __init__(self, embedding_dim=512, num_classes=None, s=64.0, m=0.50): # num_classes should be passed explicitly
        super(FaceRecognitionArcFaceModel, self).__init__()
        if num_classes is None:
            raise ValueError("num_classes must be provided for FaceRecognitionArcFaceModel")
        self.backbone = FaceRecognitionBackbone(embedding_dim=embedding_dim)
        # Pass num_classes explicitly from the argument to ArcFace
        self.arcface = ArcFace(in_features=embedding_dim, out_features=num_classes, s=s, m=m)

    def forward(self, x, label=None):
        embeddings = self.backbone(x)
        # ArcFace layer expects both embeddings and the true labels
        logits = self.arcface(embeddings, label)
        return logits

In [ ]:
import torch.nn as nn

# Ensure num_classes is available from previous steps (it was defined as 1944)
# Ensure FaceRecognitionBackbone and AdaCos classes are available from previous steps

class FaceRecognitionAdaCosModel(nn.Module):
    def __init__(self, embedding_dim=512, num_classes=None):
        super(FaceRecognitionAdaCosModel, self).__init__()
        if num_classes is None:
            raise ValueError("num_classes must be provided for FaceRecognitionAdaCosModel")
        self.backbone = FaceRecognitionBackbone(embedding_dim=embedding_dim)
        self.adacos = AdaCos(in_features=embedding_dim, out_features=num_classes)

    def forward(self, x, label=None):
        embeddings = self.backbone(x)
        # AdaCos layer expects both embeddings and the true labels
        logits = self.adacos(embeddings, label)
        return logits

### Ключевые метрики для вашего проекта:

1.  **Accuracy (Точность классификации)**: Это ваша **первоочередная метрика** для текущего задания. План требует достижения `accuracy > 0.7` как при обучении на Cross-Entropy Loss, так и при использовании ArcFace Loss. Accuracy здесь измеряет, насколько хорошо модель может *классифицировать* личности в вашем датасете. Это первый шаг для оценки того, что модель вообще учится различать людей.

2.  **Для полноценной оценки ArcFace-модели (на будущее)**: После того как вы добьётесь хорошей точности классификации, для *глубокой оценки качества эмбеддингов*, генерируемых ArcFace, вам потребуются метрики верификации, такие как:

    *   **Equal Error Rate (EER)**: Показывает точку, где частота ложных принятий (FAR) равна частоте ложных отклонений (FRR). Чем ниже EER, тем лучше модель балансирует между этими двумя типами ошибок.
    *   **TAR (True Acceptance Rate) @ FAR (False Acceptance Rate)**: Эта метрика показывает, насколько хорошо модель распознаёт истинные пары при очень низком уровне ложных срабатываний. Например, TAR@FAR=0.1% — это очень жёсткое требование для реальных систем.

Эти метрики (EER, TAR@FAR) позволят вам оценить **качество самих эмбеддингов** и их пригодность для реальных сценариев распознавания лиц, а не только для задачи классификации.

***

**Далее предлагаю перейти к обучению моделей, как это указано в вашем плане заданий!** Вы уже начали обучение ArcFace модели, поэтому давайте продолжим с ней.

In [ ]:
import torch
import torch.nn as nn

def evaluate_model(model, dataloader, loss_criterion, device):
    """
    Evaluates the given model on a specified dataset.

    Args:
        model (torch.nn.Module): The neural network model to be evaluated.
        dataloader (torch.utils.data.DataLoader): DataLoader for the evaluation dataset.
        loss_criterion (torch.nn.Module): The loss function (e.g., CrossEntropyLoss).
        device (torch.device): The device (e.g., 'cuda' or 'cpu') on which to perform evaluation.

    Returns:
        tuple: A tuple containing (average_loss, average_accuracy).
    """
    model.eval()  # Set model to evaluation mode
    eval_running_loss = 0.0
    eval_correct_predictions = 0
    eval_total_samples = 0

    with torch.no_grad():  # Disable gradient calculation during evaluation
        for batch in dataloader:
            if batch is None:
                continue

            inputs, labels = batch
            inputs = inputs.to(device)
            labels = labels.to(device).long()
            labels = labels.view(-1)

            if labels.numel() == 0:
                continue

            # Adapt forward pass for models with ArcFace/AdaCos specific layers
            if isinstance(model, FaceRecognitionArcFaceModel) or isinstance(model, FaceRecognitionAdaCosModel):
                logits = model(inputs, labels)
            else:
                logits = model(inputs)

            loss = loss_criterion(logits, labels)
            eval_running_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(logits, 1)
            eval_correct_predictions += (predicted == labels).sum().item()
            eval_total_samples += labels.size(0)

    avg_loss = eval_running_loss / eval_total_samples if eval_total_samples > 0 else 0.0
    avg_acc = eval_correct_predictions / eval_total_samples if eval_total_samples > 0 else 0.0

    return avg_loss, avg_acc

print("Defined evaluate_model function.")

Defined evaluate_model function.


In [ ]:
import torch
import torch.nn.functional as F

def extract_embeddings(model, dataloader, device):
    """
    Extracts normalized embeddings and corresponding person_id labels from a model's backbone.

    Args:
        model (torch.nn.Module): The neural network model, expected to have a 'backbone' attribute.
        dataloader (torch.utils.data.DataLoader): DataLoader for the dataset.
        device (torch.device): The device (e.g., 'cuda' or 'cpu') on which to perform inference.

    Returns:
        tuple: A tuple containing (normalized_embeddings_tensor, labels_tensor).
    """
    model.eval()  # Set model to evaluation mode
    embeddings_list = []
    labels_list = []

    with torch.no_grad():  # Disable gradient calculation
        for batch in tqdm(dataloader, leave=False, desc=f"Извлечение эмбеддингов"):
            if batch is None:
                continue

            inputs, labels = batch
            inputs = inputs.to(device)
            labels = labels.to(device).long()

            # Extract embeddings from the backbone
            # Assuming FaceRecognitionModel, FaceRecognitionArcFaceModel, or FaceRecognitionAdaCosModel
            # all have a .backbone attribute that yields embeddings.
            raw_embeddings = model.backbone(inputs)

            # Normalize the embeddings
            normalized_embeddings = F.normalize(raw_embeddings, p=2, dim=1)

            embeddings_list.append(normalized_embeddings.cpu())
            labels_list.append(labels.cpu())

    # Concatenate all embeddings and labels into single tensors
    all_embeddings = torch.cat(embeddings_list, dim=0)
    all_labels = torch.cat(labels_list, dim=0)

    return all_embeddings, all_labels

print("Defined extract_embeddings function.")

Defined extract_embeddings function.


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

def generate_pairs(embeddings, labels, num_pairs_per_class=10):
    """
    Generates positive and negative pairs from embeddings and calculates their cosine similarities.

    Args:
        embeddings (torch.Tensor): Tensor of normalized embeddings (N, embedding_dim).
        labels (torch.Tensor): Tensor of corresponding person_id labels (N,).
        num_pairs_per_class (int): Number of positive/negative pairs to generate per unique person_id.

    Returns:
        tuple: A tuple containing (similarities, is_same_person_labels),
               where similarities is a 1D tensor of cosine similarities and
               is_same_person_labels is a 1D tensor of binary labels (1 for positive, 0 for negative).
    """
    unique_labels = labels.unique()
    all_similarities = []
    all_is_same_person_labels = []

    # Convert labels to numpy for faster indexing for unique labels
    labels_np = labels.cpu().numpy()

    for i, current_label in tqdm(enumerate(unique_labels), desc=f"Создание пар"):
        # Get indices for the current person_id
        indices_current_label = torch.where(labels == current_label)[0]

        if len(indices_current_label) < 2:
            # Need at least two samples to form a positive pair
            continue

        # --- Generate Positive Pairs ---
        # Randomly select anchor and positive samples for this label
        # Ensure we don't try to get more pairs than available combinations
        num_possible_pos_pairs = len(indices_current_label) * (len(indices_current_label) - 1) // 2
        num_pos_pairs_to_generate = min(num_pairs_per_class, num_possible_pos_pairs)

        if num_pos_pairs_to_generate > 0:
            # Generate unique pairs of indices without replacement
            # Using combinations for positive pairs to avoid duplicates and ensure i!=j
            if len(indices_current_label) >= 2:
                pos_pair_indices = torch.combinations(indices_current_label, r=2)
                # Randomly sample if we have more than required
                if pos_pair_indices.shape[0] > num_pos_pairs_to_generate:
                    perm = torch.randperm(pos_pair_indices.shape[0])
                    pos_pair_indices = pos_pair_indices[perm[:num_pos_pairs_to_generate]]

                anchor_embeddings_pos = embeddings[pos_pair_indices[:, 0]]
                positive_embeddings = embeddings[pos_pair_indices[:, 1]]
                pos_similarities = F.cosine_similarity(anchor_embeddings_pos, positive_embeddings)
                all_similarities.append(pos_similarities)
                all_is_same_person_labels.append(torch.ones_like(pos_similarities))


        # --- Generate Negative Pairs ---
        # Randomly select anchor from current label and negative from a different label
        other_labels = unique_labels[unique_labels != current_label]
        if len(other_labels) == 0: # No other labels to form negative pairs
            continue

        num_neg_pairs_to_generate = num_pairs_per_class # Generate same number of negative pairs

        if num_neg_pairs_to_generate > 0:
            # Sample anchors from current_label
            if len(indices_current_label) == 0: # Should be caught by positive pair check, but good to have
                continue

            anchor_indices_neg = indices_current_label[torch.randint(0, len(indices_current_label), (num_neg_pairs_to_generate,))]
            anchor_embeddings_neg = embeddings[anchor_indices_neg]

            # Sample negative samples from other labels
            random_other_labels_indices = torch.randint(0, len(other_labels), (num_neg_pairs_to_generate,))
            random_other_labels = other_labels[random_other_labels_indices]

            negative_indices = []
            for other_l in random_other_labels:
                indices_other_label = torch.where(labels == other_l)[0]
                if len(indices_other_label) > 0:
                    negative_indices.append(indices_other_label[torch.randint(0, len(indices_other_label), (1,))])
                else:
                    # Fallback if no samples for selected 'other_l', rare but possible if dataset is sparse
                    negative_indices.append(indices_current_label[torch.randint(0, len(indices_current_label), (1,))]) # Self-pair as placeholder, will be filtered

            if negative_indices:
                negative_embeddings = embeddings[torch.cat(negative_indices).flatten()]
                neg_similarities = F.cosine_similarity(anchor_embeddings_neg, negative_embeddings)
                all_similarities.append(neg_similarities)
                all_is_same_person_labels.append(torch.zeros_like(neg_similarities))

    if not all_similarities:
        # Handle case where no pairs could be generated
        return torch.tensor([]), torch.tensor([])

    final_similarities = torch.cat(all_similarities)
    final_is_same_person_labels = torch.cat(all_is_same_person_labels)

    return final_similarities, final_is_same_person_labels

print("Defined generate_pairs function.")


Defined generate_pairs function.


In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, auc
from scipy.optimize import brentq
from scipy.interpolate import interp1d

def calculate_eer(similarities, is_same_person_labels):
    """
    Calculates Equal Error Rate (EER) and ROC AUC score.

    Args:
        similarities (torch.Tensor or np.ndarray): 1D array of cosine similarities.
        is_same_person_labels (torch.Tensor or np.ndarray): 1D array of binary labels (1 for same person, 0 for different).

    Returns:
        tuple: A tuple containing (eer, roc_auc, fpr, tpr, thresholds).
    """
    # Ensure inputs are numpy arrays for sklearn functions
    if isinstance(similarities, torch.Tensor):
        similarities = similarities.cpu().numpy()
    if isinstance(is_same_person_labels, torch.Tensor):
        is_same_person_labels = is_same_person_labels.cpu().numpy()

    # Calculate False Positive Rate (FPR) and True Positive Rate (TPR)
    fpr, tpr, thresholds = roc_curve(is_same_person_labels, similarities)

    # Calculate AUC
    roc_auc = auc(fpr, tpr)

    # Calculate EER
    # EER is the point where FPR == (1 - TPR) or FPR + TPR = 1
    # EER is the rate at which both errors are equal. Interpolate to find this point.
    eer = brentq(lambda x : 1. - x - interp1d(fpr, tpr)(x), 0., 1.)

    return eer, roc_auc, fpr, tpr, thresholds

print("Defined calculate_eer function.")

Defined calculate_eer function.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from tqdm.notebook import tqdm # Импортируем tqdm для отображения прогресса
import torch.optim.lr_scheduler as lr_scheduler # Импортируем планировщик скорости обучения

def train_model(model, train_dataloader, val_dataloader, optimizer, loss_criterion, num_epochs, display_freq, device, patience=5):
    """
    Обучает заданную модель.

    Аргументы:
        model (torch.nn.Module): Обучаемая нейронная сеть.
        train_dataloader (torch.utils.data.DataLoader): DataLoader для обучающего набора данных.
        val_dataloader (torch.utils.data.DataLoader): DataLoader для валидационного набора данных.
        optimizer (torch.optim.Optimizer): Алгоритм оптимизации (например, Adam, SGD).
        loss_criterion (torch.nn.Module): Функция потерь (например, CrossEntropyLoss).
        num_epochs (int): Общее количество эпох для обучения.
        display_freq (int): Как часто (в эпохах) выводить прогресс обучения и результаты валидации.
        device (torch.device): Устройство (например, 'cuda' или 'cpu'), на котором выполняется обучение.
        patience (int, optional): Количество эпох для ожидания улучшения перед остановкой. По умолчанию: None (без ранней остановки).
    """

    torch.cuda.empty_cache()

    history = {'train_loss': [],
               'val_loss': [],
               'train_acc': [],
               'val_acc': [],
               'val_eer': [],
               'val_roc_auc': []
               }

    model.to(device)

    # Настройка планировщика скорости обучения ReduceLROnPlateau
    # Отслеживаем валидационную точность, уменьшаем LR в 2 раза, если точность не улучшается 3 эпохи.
    scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.3, patience=3)

    # Настройка ранней остановки
    best_val_acc = -1
    epochs_no_improve = 0
    early_stop = False
    best_model_state = None # Для сохранения лучшей модели

    for epoch in range(num_epochs):
        model.train()  # Переводим модель в режим обучения
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        # Оборачиваем train_dataloader в tqdm для отображения прогресса
        train_loop = tqdm(train_dataloader, leave=False, desc=f"Эпоха {epoch+1}/{num_epochs} (Обучение)")
        for batch in train_loop: # Итерируемся по батчам
            if batch is None:
                # Этот батч был полностью отфильтрован custom_collate_fn
                continue

            inputs, labels = batch # Ожидаем (изображение, метка) от FaceRecognitionDataset в режиме 'face_recognition'
            inputs = inputs.to(device)
            labels = labels.to(device).long() # Убеждаемся, что метки имеют тип LongTensor
            labels = labels.view(-1) # Выравниваем метки до 1D

            # Пропускаем батч, если метки пусты после обработки (в идеале не должно происходить, если батч не None, но для подстраховки)
            if labels.numel() == 0:
                continue

            optimizer.zero_grad()

            # Обрабатываем модели со специфическим прямым проходом для ArcFace/AdaCos
            if isinstance(model, FaceRecognitionArcFaceModel) or isinstance(model, FaceRecognitionAdaCosModel):
                logits = model(inputs, labels)
            else:
                logits = model(inputs)

            loss = loss_criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(logits, 1)
            correct_predictions += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            # Обновляем полосу прогресса tqdm с текущим значением потерь
            train_loop.set_postfix(loss=running_loss/total_samples, acc=correct_predictions/total_samples)

        epoch_train_loss = running_loss / total_samples if total_samples > 0 else 0.0
        epoch_train_acc = correct_predictions / total_samples if total_samples > 0 else 0.0
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)

        # Фаза валидации
        model.eval()  # Переводим модель в режим оценки
        val_running_loss = 0.0
        val_correct_predictions = 0
        val_total_samples = 0

        with torch.no_grad():  # Отключаем вычисление градиентов во время валидации
            # Оборачиваем val_dataloader в tqdm для отображения прогресса
            val_loop = tqdm(val_dataloader, leave=False, desc=f"Эпоха {epoch+1}/{num_epochs} (Валидация)")
            for batch in val_loop: # Итерируемся по батчам
                if batch is None:
                    # Этот батч был полностью отфильтрован custom_collate_fn
                    continue

                inputs, labels = batch # Ожидаем (изображение, метка) от FaceRecognitionDataset в режиме 'face_recognition'
                inputs = inputs.to(device)
                labels = labels.to(device).long() # Убеждаемся, что метки имеют тип LongTensor
                labels = labels.view(-1) # Выравниваем метки до 1D

                # Пропускаем батч, если метки пусты после обработки
                if labels.numel() == 0:
                    continue

                logits = model(inputs)

                loss = loss_criterion(logits, labels)
                val_running_loss += loss.item() * inputs.size(0)

                _, predicted = torch.max(logits, 1)
                val_correct_predictions += (predicted == labels).sum().item()
                val_total_samples += labels.size(0)

                # Обновляем полосу прогресса tqdm с текущим значением потерь валидации
                val_loop.set_postfix(loss=val_running_loss/val_total_samples, acc=val_correct_predictions/val_total_samples)

        epoch_val_loss = val_running_loss / val_total_samples if val_total_samples > 0 else 0.0
        epoch_val_acc = val_correct_predictions / val_total_samples if val_total_samples > 0 else 0.0
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)

        # Шаг планировщика скорости обучения (scheduler step)
        scheduler.step(epoch_val_acc)

        # Проверка ранней остановки
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            epochs_no_improve = 0
            best_model_state = model.state_dict() # Сохраняем состояние лучшей модели
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Ранняя остановка сработала после {epochs_no_improve} эпох без улучшения валидационной метрики.")
                early_stop = True

        # Выводим прогресс и рассчитываем EER/ROC AUC, если достигнута частота отображения
        if (epoch + 1) % display_freq == 0:
            print(f"Эпоха [{epoch+1}/{num_epochs}]\n "
                  f"Лосс на обучении: {epoch_train_loss:.4f}, Точность на обучении: {epoch_train_acc:.4f}\n "
                  f"Лосс на валидации: {epoch_val_loss:.4f}, Точность на валидации: {epoch_val_acc:.4f}")

            # --- Рассчитываем EER и ROC AUC для валидационного набора ---
            if val_total_samples > 0:
                # 1. Извлекаем эмбеддинги
                val_embeddings, val_labels_eer = extract_embeddings(model, val_dataloader, device)

                # 2. Генерируем пары
                val_similarities, val_is_same_person_labels = generate_pairs(val_embeddings, val_labels_eer, num_pairs_per_class=10)

                # 3. Рассчитываем EER и ROC AUC
                if len(val_similarities) > 0:
                    val_eer, val_roc_auc, _, _, _ = calculate_eer(val_similarities, val_is_same_person_labels)
                    history['val_eer'].append(val_eer)
                    history['val_roc_auc'].append(val_roc_auc)
                    print(f"Val EER: {val_eer:.4f}, Val ROC AUC: {val_roc_auc:.4f}\n")
                else:
                    print("Не удалось сгенерировать достаточно пар для расчета EER/ROC AUC на валидационном наборе.\n")
                    history['val_eer'].append(float('nan'))
                    history['val_roc_auc'].append(float('nan'))
            else:
                print("Не обработаны валидационные сэмплы для расчета EER/ROC AUC.\n")
                history['val_eer'].append(float('nan'))
                history['val_roc_auc'].append(float('nan'))

        if early_stop:
            # Восстанавливаем лучшую модель перед выходом
            print("Восстановление лучшей модели...")
            model.load_state_dict(best_model_state)
            break

    return history

In [ ]:
import torch.optim as optim

# Instantiate the FaceRecognitionModel (linear classifier)
# num_classes is already defined globally (2944)
num_classes = train_dataloader.dataset.num_classes
linear_model = FaceRecognitionModel(embedding_dim=512, num_classes=num_classes)

# Define optimizer and loss criterion for the linear model
optimizer_linear = optim.AdamW(linear_model.parameters(), lr=0.001)
loss_criterion_linear = nn.CrossEntropyLoss()

# Set training parameters
num_epochs = 100 # You can adjust this
display_freq = 1 # Display results every epoch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
patience = 10 # Early stopping patience

# Run training for the linear model
linear_history = train_model(
    linear_model,
    train_dataloader,
    val_dataloader,
    optimizer_linear,
    loss_criterion_linear,
    num_epochs,
    display_freq,
    device,
    patience=patience
)

Эпоха 1/100 (Обучение):   0%|          | 0/167 [00:00<?, ?it/s]

Эпоха 1/100 (Валидация):   0%|          | 0/38 [00:00<?, ?it/s]

Эпоха [1/100]
 Лосс на обучении: 6.2285, Точность на обучении: 0.0569
 Лосс на валидации: 4.4727, Точность на валидации: 0.1751


Извлечение эмбеддингов:   0%|          | 0/38 [00:00<?, ?it/s]

Создание пар: 0it [00:00, ?it/s]

...

Эпоха 48/100 (Валидация):   0%|          | 0/38 [00:00<?, ?it/s]

Ранняя остановка сработала после 10 эпох без улучшения валидационной метрики.
Эпоха [48/100]
 Лосс на обучении: 0.0013, Точность на обучении: 0.9997
 Лосс на валидации: 0.6300, Точность на валидации: 0.8983


Извлечение эмбеддингов:   0%|          | 0/38 [00:00<?, ?it/s]

Создание пар: 0it [00:00, ?it/s]

Val EER: 0.0679, Val ROC AUC: 0.9783

Восстановление лучшей модели...


In [ ]:
torch.save(linear_model.state_dict(), 'PROJECT/FaceAlignment/FR_linear.pth')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
linear_model.load_state_dict(torch.load('PROJECT/FaceAlignment/FR_linear.pth'))
# Ensure the linear_model is on the correct device and loss criterion is defined
linear_model.to(device)

# Evaluate the Linear model on the test dataset
test_embeddings, test_labels_eer = extract_embeddings(linear_model, test_dataloader, device)
test_similarities, test_is_same_person_labels = generate_pairs(test_embeddings, test_labels_eer, num_pairs_per_class=10)
test_eer, test_roc_auc, _, _, _ = calculate_eer(test_similarities, test_is_same_person_labels)

print(f"Test ROC AUC: {test_roc_auc:.4f}")
print(f"Test EER: {test_eer:.4f}")

Извлечение эмбеддингов:   0%|          | 0/20 [00:00<?, ?it/s]

Создание пар: 0it [00:00, ?it/s]

Test ROC AUC: 0.9609
Test EER: 0.0958




---



In [ ]:
import torch.optim as optim

# Instantiate the FaceRecognitionModel (linear classifier)
# num_classes is already defined globally (2944)
num_classes = train_dataloader.dataset.num_classes
ArcFace_model = FaceRecognitionArcFaceModel(embedding_dim=512, num_classes=num_classes, s=64.0, m=0.4)

# Define optimizer and loss criterion for the linear model
optimizer_ArcFace = optim.AdamW(ArcFace_model.parameters(), lr=1e-3)
loss_criterion_linear = nn.CrossEntropyLoss()

# Set training parameters
num_epochs = 100 # You can adjust this
display_freq = 1 # Display results every epoch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
patience = 10 # Early stopping patience

# Run training for the linear model
ArcFace_history = train_model(
    ArcFace_model,
    train_dataloader,
    val_dataloader,
    optimizer_ArcFace,
    loss_criterion_linear,
    num_epochs,
    display_freq,
    device,
    patience=patience
)

Эпоха 1/100 (Обучение):   0%|          | 0/167 [00:00<?, ?it/s]

Эпоха 1/100 (Валидация):   0%|          | 0/38 [00:00<?, ?it/s]

Эпоха [1/100]
 Лосс на обучении: 33.5865, Точность на обучении: 0.0000
 Лосс на валидации: 7.1867, Точность на валидации: 0.0186


Извлечение эмбеддингов:   0%|          | 0/38 [00:00<?, ?it/s]

Создание пар: 0it [00:00, ?it/s]

...

Эпоха 48/100 (Валидация):   0%|          | 0/38 [00:00<?, ?it/s]

Ранняя остановка сработала после 10 эпох без улучшения валидационной метрики.
Эпоха [48/100]
 Лосс на обучении: 0.0424, Точность на обучении: 0.9933
 Лосс на валидации: 0.8280, Точность на валидации: 0.9319


Извлечение эмбеддингов:   0%|          | 0/38 [00:00<?, ?it/s]

Создание пар: 0it [00:00, ?it/s]

Val EER: 0.0509, Val ROC AUC: 0.9834

Восстановление лучшей модели...


In [ ]:
torch.save(ArcFace_model.state_dict(), 'PROJECT/FaceAlignment/FR_ArcFace.pth')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ArcFace_model.load_state_dict(torch.load('PROJECT/FaceAlignment/FR_ArcFace.pth'))
# Ensure the linear_model is on the correct device and loss criterion is defined
ArcFace_model.to(device)

# Evaluate the Linear model on the test dataset
test_embeddings, test_labels_eer = extract_embeddings(ArcFace_model, test_dataloader, device)
test_similarities, test_is_same_person_labels = generate_pairs(test_embeddings, test_labels_eer, num_pairs_per_class=10)
test_eer, test_roc_auc, _, _, _ = calculate_eer(test_similarities, test_is_same_person_labels)

print(f"Test ROC AUC: {test_roc_auc:.4f}")
print(f"Test EER: {test_eer:.4f}")

Извлечение эмбеддингов:   0%|          | 0/20 [00:00<?, ?it/s]

Создание пар: 0it [00:00, ?it/s]

Test ROC AUC: 0.9530
Test EER: 0.1082




---



In [ ]:
import torch.optim as optim

# Instantiate the FaceRecognitionModel (linear classifier)
# num_classes is already defined globally (2944)
num_classes = train_dataloader.dataset.num_classes
AdaCos_model = FaceRecognitionAdaCosModel(embedding_dim=512, num_classes=num_classes)

# Define optimizer and loss criterion for the linear model
optimizer_AdaCos = optim.AdamW(AdaCos_model.parameters(), lr=1e-3)
loss_criterion_linear = nn.CrossEntropyLoss()
# Set training parameters
num_epochs = 100 # You can adjust this
display_freq = 1 # Display results every epoch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
patience = 10 # Early stopping patience

# Run training for the linear model
AdaCos_history = train_model(
    AdaCos_model,
    train_dataloader,
    val_dataloader,
    optimizer_AdaCos,
    loss_criterion_linear,
    num_epochs,
    display_freq,
    device,
    patience=patience
)

Эпоха 1/100 (Обучение):   0%|          | 0/167 [00:00<?, ?it/s]

Эпоха 1/100 (Валидация):   0%|          | 0/38 [00:00<?, ?it/s]

Эпоха [1/100]
 Лосс на обучении: 6.6555, Точность на обучении: 0.0315
 Лосс на валидации: 5.4528, Точность на валидации: 0.1233


Извлечение эмбеддингов:   0%|          | 0/38 [00:00<?, ?it/s]

Создание пар: 0it [00:00, ?it/s]

...

Эпоха 57/100 (Валидация):   0%|          | 0/38 [00:00<?, ?it/s]

Ранняя остановка сработала после 10 эпох без улучшения валидационной метрики.
Эпоха [57/100]
 Лосс на обучении: 0.7689, Точность на обучении: 0.9991
 Лосс на валидации: 1.4685, Точность на валидации: 0.8854


Извлечение эмбеддингов:   0%|          | 0/38 [00:00<?, ?it/s]

Создание пар: 0it [00:00, ?it/s]

Val EER: 0.0658, Val ROC AUC: 0.9661

Восстановление лучшей модели...


In [ ]:
torch.save(AdaCos_model.state_dict(), 'PROJECT/FaceAlignment/FR_AdaCos.pth')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AdaCos_model.load_state_dict(torch.load('PROJECT/FaceAlignment/FR_AdaCos.pth'))
# Ensure the linear_model is on the correct device and loss criterion is defined
AdaCos_model.to(device)

# Evaluate the Linear model on the test dataset
test_embeddings, test_labels_eer = extract_embeddings(AdaCos_model, test_dataloader, device)
test_similarities, test_is_same_person_labels = generate_pairs(test_embeddings, test_labels_eer, num_pairs_per_class=10)
test_eer, test_roc_auc, _, _, _ = calculate_eer(test_similarities, test_is_same_person_labels)

print(f"Test ROC AUC: {test_roc_auc:.4f}")
print(f"Test EER: {test_eer:.4f}")

Извлечение эмбеддингов:   0%|          | 0/20 [00:00<?, ?it/s]

Создание пар: 0it [00:00, ?it/s]

Test ROC AUC: 0.8049
Test EER: 0.2546
